# NB08_whatif — Mô phỏng What-If (mục 5.5)
*Phụ trách: Huy*

Đầu vào: `menu_matrix.csv` (NB06), `bom.csv`, `ingredient.csv` đặt cùng thư mục làm việc.
Đầu ra: `processed/whatif.csv`, `whatif_summary.csv`, `whatif_sensitivity.csv`, `charts/whatif_sensitivity.png`.

Mô hình: GP' = Σ(P'i − C'i) × Q'i (mục 2.6.7). KB1: Plowhorse +5% giá, −2% lượng; KB2: giá sữa −10%; KB3: loại 3 món Dog; độ nhạy giá −5% → +10%, bước 2,5%.

In [ ]:
# ==== NB08_whatif — Mô phỏng What-If (mục 5.5) ====
# Đầu vào: processed/menu_matrix.csv (NB06, cấp toàn chuỗi, ngưỡng 100%/n), bom.csv, ingredient.csv
# Đầu ra : processed/whatif.csv (từng món × kịch bản), processed/whatif_summary.csv,
#          processed/whatif_sensitivity.csv, charts/whatif_sensitivity.png
# Mô hình (mục 2.6.7): GP' = Σ (P'i − C'i) × Q'i ;  P' = P(1+ΔP), C' = C(1+ΔC), Q' = Q(1+ΔQ)
import os
import numpy as np
import pandas as pd

BASE = os.environ.get('NB08_BASE', '.')
OUT = os.path.join(BASE, 'processed'); CH = os.path.join(BASE, 'charts')
os.makedirs(OUT, exist_ok=True); os.makedirs(CH, exist_ok=True)
rd = lambda f: pd.read_csv(os.path.join(BASE, f), encoding='utf-8-sig')

m = rd('menu_matrix.csv')
base = m[m['scope_type'] == 'chain'][['product_id', 'product_detail', 'product_category', 'Q', 'P', 'std_cost', 'class_100']].copy()
base = base.rename(columns={'std_cost': 'C', 'class_100': 'group'}).reset_index(drop=True)
assert len(base) == 41

bom, ing = rd('bom.csv'), rd('ingredient.csv')
b = bom.merge(ing, on='ingredient_id')
b['line_cost'] = b['quantity_ep'] / (b['yield_pct'] / 100) * b['unit_cost_usd']          # chi phí theo AP
assert np.allclose(b.groupby('product_id')['line_cost'].sum().reindex(base.product_id).values, base['C'].values)
MILK_ID = 'I008'                                                                            # Fresh Milk
base['milk_cost'] = base['product_id'].map(b[b.ingredient_id == MILK_ID].groupby('product_id')['line_cost'].sum()).fillna(0.0)

DOGS = [28, 87, 34]          # 3 món Dog có total_cm thấp nhất (NB06, whatif_dogs.csv)
ELASTICITY = -0.4            # KB1: giá +5% → lượng −2%  ⇒  e = −2% / 5%

def run(name, dP=None, dC_milk=0.0, drop=(), e=ELASTICITY):
    """dP: dict product_id -> % thay đổi giá; dC_milk: % thay đổi giá sữa; drop: các món loại bỏ."""
    d = base.copy()
    d['dP'] = d['product_id'].map(dP or {}).fillna(0.0)
    d['dQ'] = e * d['dP']
    d.loc[d.product_id.isin(drop), 'dQ'] = -1.0
    d['P_new'] = d['P'] * (1 + d['dP'])
    d['C_new'] = d['C'] + d['milk_cost'] * dC_milk
    d['Q_new'] = d['Q'] * (1 + d['dQ'])
    d['CM'] = d['P'] - d['C']; d['CM_new'] = d['P_new'] - d['C_new']
    d['GP'] = d['CM'] * d['Q']; d['GP_new'] = d['CM_new'] * d['Q_new']
    d['REV'] = d['P'] * d['Q']; d['REV_new'] = d['P_new'] * d['Q_new']
    d['COST'] = d['C'] * d['Q']; d['COST_new'] = d['C_new'] * d['Q_new']
    d.insert(0, 'scenario', name)
    return d


In [ ]:
plow = base.loc[base.group == 'Plowhorses', 'product_id'].tolist()
scen = {
    'KB0 – Hiện trạng': run('KB0 – Hiện trạng'),
    'KB1 – Plowhorse +5% giá, −2% lượng': run('KB1 – Plowhorse +5% giá, −2% lượng', dP={p: 0.05 for p in plow}),
    'KB2 – Giá sữa −10%': run('KB2 – Giá sữa −10%', dC_milk=-0.10),
    'KB3 – Loại 3 món Dog': run('KB3 – Loại 3 món Dog', drop=DOGS),
}
detail = pd.concat(scen.values(), ignore_index=True)
detail.to_csv(os.path.join(OUT, 'whatif.csv'), index=False, encoding='utf-8-sig')

def summ(d):
    return pd.Series({'Q': d.Q_new.sum(), 'Doanh thu': d.REV_new.sum(), 'Giá vốn định mức': d.COST_new.sum(),
                      'Lợi nhuận gộp (GP)': d.GP_new.sum(), 'Food Cost %': 100 * d.COST_new.sum() / d.REV_new.sum()})
summary = pd.DataFrame({k: summ(v) for k, v in scen.items()}).T
gp0 = summary.loc['KB0 – Hiện trạng', 'Lợi nhuận gộp (GP)']
summary['ΔGP (USD)'] = summary['Lợi nhuận gộp (GP)'] - gp0
summary['ΔGP (%)'] = 100 * summary['ΔGP (USD)'] / gp0
summary.to_csv(os.path.join(OUT, 'whatif_summary.csv'), encoding='utf-8-sig')


In [ ]:
# ---- Phân tích độ nhạy: giá nhóm Plowhorse từ −5% đến +10%, bước 2,5%, với 4 mức co giãn ----
steps = np.arange(-0.05, 0.10001, 0.025)
sens = []
for e in (0.0, -0.4, -1.0, -2.0):
    for s in steps:
        d = run('sens', dP={p: s for p in plow}, e=e)
        sens.append({'elasticity': e, 'dP_pct': round(100 * s, 1), 'GP': d.GP_new.sum(), 'dGP': d.GP_new.sum() - gp0})
sens = pd.DataFrame(sens)
sens.to_csv(os.path.join(OUT, 'whatif_sensitivity.csv'), index=False, encoding='utf-8-sig')

# Độ co giãn hòa vốn của KB1: Σ[(1.05P − C)(1 + 0.05e) − (P − C)]Q = 0
pl = base[base.product_id.isin(plow)]
A = ((1.05 * pl.P - pl.C) * pl.Q).sum(); B0 = ((pl.P - pl.C) * pl.Q).sum()
e_be = (B0 / A - 1) / 0.05
# Sữa: mỗi 1% thay đổi giá sữa làm GP đổi bao nhiêu
milk_per_pct = (base.milk_cost * base.Q).sum() / 100
# KB3: tỷ lệ khách chuyển sang món khác để bù phần CM mất đi
rest = base[~base.product_id.isin(DOGS)]
cm_rest = ((rest.P - rest.C) * rest.Q).sum() / rest.Q.sum()
lost_q = base[base.product_id.isin(DOGS)].Q.sum()
lost_cm = (base[base.product_id.isin(DOGS)].eval('(P - C) * Q')).sum()
switch_be = lost_cm / (lost_q * cm_rest)


In [ ]:
import matplotlib; matplotlib.use('Agg')


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4.6))
labels = {0.0: 'Lượng bán không đổi (e = 0)', -0.4: 'e = −0,4 (giả định KB1)', -1.0: 'e = −1', -2.0: 'e = −2'}
styles = {0.0: ('#1f77b4', '-'), -0.4: ('#d62728', '-'), -1.0: ('#7f7f7f', '--'), -2.0: ('#7f7f7f', ':')}
for e, g in sens.groupby('elasticity', sort=False):
    col, ls = styles[e]
    ax.plot(g.dP_pct, g.dGP, marker='o', color=col, ls=ls, lw=2 if e == -0.4 else 1.5, label=labels[e])
ax.axhline(0, color='black', lw=0.8); ax.axvline(0, color='black', lw=0.5, alpha=.4)
ax.set_xlabel('Thay đổi giá bán nhóm Plowhorse (%)'); ax.set_ylabel('Thay đổi lợi nhuận gộp toàn chuỗi (USD)')
ax.set_xticks(steps * 100)
ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f'{v:,.0f}'.replace(',', '.')))
ax.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f'{v:g}'.replace('.', ',')))
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=.25)
for s in ('top', 'right'): ax.spines[s].set_visible(False)
fig.tight_layout(); fig.savefig(os.path.join(CH, 'whatif_sensitivity.png'), dpi=200)

pd.set_option('display.width', 200)
print(summary.round(2).to_string())
print('\nĐộ nhạy:\n', sens.pivot(index='dP_pct', columns='elasticity', values='dGP').round(2).to_string())
print(f'\nPlowhorse: {len(plow)} món; độ co giãn hòa vốn KB1 e* = {e_be:.3f}')
print(f'Sữa: {int((base.milk_cost > 0).sum())} món dùng sữa; mỗi 1% giá sữa = {milk_per_pct:.2f} USD GP; tỷ trọng sữa trong giá vốn = {100 * (base.milk_cost * base.Q).sum() / (base.C * base.Q).sum():.2f}%')
print(f'KB3: mất {lost_q} ly, {lost_cm:.2f} USD CM; CM bình quân món còn lại {cm_rest:.4f}; tỷ lệ chuyển hòa vốn {100 * switch_be:.1f}%')
print('Món dùng sữa:', base[base.milk_cost > 0].product_detail.tolist())
